<a href="https://colab.research.google.com/github/aditibhat017-prog/cognitivefunction/blob/main/notebooks/02_replication_2011_2012.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Replication Analysis — NHANES 2011–2012

## Objective

This notebook evaluates whether the association observed in the NHANES 2013–2014 cohort between wearable-derived mean daily physical activity and cognitive performance can be replicated in an independent NHANES 2011–2012 cohort.

### Primary Research Question

Does higher wearable-measured mean daily physical activity remain associated with higher Digit Symbol Substitution Test (DSST) performance after adjustment for age, sex, and education?

### Primary Hypothesis

Higher mean daily wearable-measured physical activity will be associated with higher DSST performance after accounting for age, sex, and education.

### Replication Strategy

The same data-cleaning rules, wearable validity criteria, feature-engineering procedures, and statistical models used for the NHANES 2013–2014 cohort will be applied to the NHANES 2011–2012 cohort.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import statsmodels.formula.api as smf

## 2. Download NHANES 2011–2012 Data

In [2]:
import os
import requests

os.makedirs("data/raw", exist_ok=True)

files_2011_2012 = {
    "CFQ_G.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/CFQ_G.xpt",
    "PAXDAY_G.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/PAXDAY_G.xpt",
    "DEMO_G.xpt": "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2011/DataFiles/DEMO_G.xpt"
}

for filename, url in files_2011_2012.items():
    response = requests.get(url)
    response.raise_for_status()

    with open(f"data/raw/{filename}", "wb") as f:
        f.write(response.content)

    print(f"Downloaded: {filename}")

Downloaded: CFQ_G.xpt
Downloaded: PAXDAY_G.xpt
Downloaded: DEMO_G.xpt


In [3]:
from pathlib import Path

for file in Path("data/raw").glob("*.xpt"):
    print(
        file.name,
        f"{file.stat().st_size / 1_000_000:.2f} MB"
    )

PAXDAY_G.xpt 6.49 MB
DEMO_G.xpt 3.75 MB
CFQ_G.xpt 0.26 MB


## 3. Load Data

In [4]:
cfq_g = pd.read_sas("data/raw/CFQ_G.xpt")
pax_g = pd.read_sas("data/raw/PAXDAY_G.xpt")
demo_g = pd.read_sas("data/raw/DEMO_G.xpt")

print("Cognitive:", cfq_g.shape)
print("Activity:", pax_g.shape)
print("Demographics:", demo_g.shape)

Cognitive: (1687, 19)
Activity: (61168, 15)
Demographics: (9756, 48)


In [5]:
print("Cognitive columns:")
print(cfq_g.columns.tolist())

print("\nActivity columns:")
print(pax_g.columns.tolist())

print("\nDemographic columns:")
print(demo_g.columns.tolist())

Cognitive columns:
['SEQN', 'CFASTAT', 'CFALANG', 'CFDCCS', 'CFDCRNC', 'CFDCST1', 'CFDCST2', 'CFDCST3', 'CFDCSR', 'CFDCIT1', 'CFDCIT2', 'CFDCIT3', 'CFDCIR', 'CFDAPP', 'CFDARNC', 'CFDAST', 'CFDDPP', 'CFDDRNC', 'CFDDS']

Activity columns:
['SEQN', 'PAXDAYD', 'PAXDAYWD', 'PAXSSNDP', 'PAXMSTD', 'PAXTMD', 'PAXAISMD', 'PAXVMD', 'PAXMTSD', 'PAXWWMD', 'PAXSWMD', 'PAXNWMD', 'PAXUMD', 'PAXLXSD', 'PAXQFD']

Demographic columns:
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGY', 'RIDEXAGM', 'DMQMILIZ', 'DMQADFC', 'DMDBORN4', 'DMDCITZN', 'DMDYRSUS', 'DMDEDUC3', 'DMDEDUC2', 'DMDMARTL', 'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA', 'INDHHIN2', 'INDFMIN2', 'INDFMPIR', 'DMDHHSIZ', 'DMDFMSIZ', 'DMDHHSZA', 'DMDHHSZB', 'DMDHHSZE', 'DMDHRGND', 'DMDHRAGE', 'DMDHRBR4', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU']


In [6]:
cognitive_vars = [
    "SEQN",
    "CFDDS",
    "CFDAST",
    "CFDCSR",
    "CFDCST1",
    "CFDCST2",
    "CFDCST3"
]

wearable_vars = [
    "SEQN",
    "PAXDAYD",
    "PAXMTSD",
    "PAXVMD",
    "PAXWWMD",
    "PAXSWMD",
    "PAXNWMD",
    "PAXUMD",
    "PAXQFD"
]

demo_vars = [
    "SEQN",
    "RIDAGEYR",
    "RIAGENDR",
    "DMDEDUC2"
]

In [7]:
print("Missing cognitive variables:",
      [x for x in cognitive_vars if x not in cfq_g.columns])

print("Missing wearable variables:",
      [x for x in wearable_vars if x not in pax_g.columns])

print("Missing demographic variables:",
      [x for x in demo_vars if x not in demo_g.columns])

Missing cognitive variables: []
Missing wearable variables: []
Missing demographic variables: []


## 4. Data Cleaning

In [8]:
cfq_clean_g = cfq_g.copy()
pax_clean_g = pax_g.copy()
demo_clean_g = demo_g.copy()

# Convert extremely small SAS missing-value representations to NaN
for df in [cfq_clean_g, pax_clean_g, demo_clean_g]:
    numeric_cols = df.select_dtypes(include="number").columns

    for col in numeric_cols:
        df.loc[df[col].abs() < 1e-50, col] = np.nan

In [9]:
# NHANES education:
# 7 = Refused
# 9 = Don't know
demo_clean_g["DMDEDUC2"] = (
    demo_clean_g["DMDEDUC2"]
    .replace([7, 9], np.nan)
)

In [10]:
pax_clean_g["PAXDAYD"] = (
    pax_clean_g["PAXDAYD"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype(float)
)

In [11]:
pax_clean_g["PAXDAYD"].value_counts().sort_index()

,count
PAXDAYD,
1.0,6917
2.0,6907
3.0,6884
4.0,6855
5.0,6818
6.0,6770
7.0,6733
8.0,6676
9.0,6608


## 5. Wearable Quality Control

In [12]:
# Same threshold used in the 2013–2014 analysis
pax_clean_g["valid_day"] = (
    pax_clean_g["PAXVMD"] >= 600
)

pax_clean_g["valid_day"].value_counts()

,count
valid_day,
True,57449
False,3719


In [13]:
valid_days_g = (
    pax_clean_g
    .groupby("SEQN")["valid_day"]
    .sum()
)

valid_days_g.describe()

,valid_day
count,6917.000000
mean,8.305479
std,1.054166
min,0.000000
25%,8.000000
50%,8.000000
75%,9.000000
max,9.000000


In [14]:
valid_days_g.value_counts().sort_index()

,count
valid_day,
0,14
1,19
2,31
3,36
4,32
5,55
6,57
7,61
8,3420


## 6. Participant-Level Wearable Feature Engineering

In [15]:
valid_pax_g = (
    pax_clean_g[
        pax_clean_g["valid_day"]
    ]
    .copy()
)

wearable_features_g = (
    valid_pax_g
    .groupby("SEQN")
    .agg(
        mean_daily_activity=("PAXMTSD", "mean"),
        activity_sd=("PAXMTSD", "std"),
        mean_valid_minutes=("PAXVMD", "mean"),
        mean_wake_wear_minutes=("PAXWWMD", "mean"),
        mean_sleep_wear_minutes=("PAXSWMD", "mean"),
        mean_nonwear_minutes=("PAXNWMD", "mean"),
        mean_unknown_minutes=("PAXUMD", "mean"),
        valid_days=("valid_day", "sum")
    )
    .reset_index()
)

In [16]:
wearable_features_g["activity_cv"] = (
    wearable_features_g["activity_sd"] /
    wearable_features_g["mean_daily_activity"]
)

In [17]:
wearable_features_g = wearable_features_g[
    wearable_features_g["valid_days"] >= 4
].copy()

print("Wearable analysis sample:",
      wearable_features_g.shape)

Wearable analysis sample: (6817, 10)


## 7. Merge Cognitive, Demographic, and Wearable Data

In [18]:
cognitive_analysis_g = (
    cfq_clean_g[cognitive_vars]
    .copy()
)

demo_analysis_g = (
    demo_clean_g[demo_vars]
    .copy()
)

analysis_g = (
    cognitive_analysis_g
    .merge(
        demo_analysis_g,
        on="SEQN",
        how="left"
    )
    .merge(
        wearable_features_g,
        on="SEQN",
        how="inner"
    )
)

In [19]:
analysis_g = analysis_g[
    analysis_g["RIDAGEYR"] >= 60
].copy()

In [20]:
dsst_g = analysis_g.dropna(
    subset=[
        "CFDDS",
        "DMDEDUC2"
    ]
).copy()

In [21]:
print("Participants:", len(dsst_g))

print("\nAge range:")
print(
    dsst_g["RIDAGEYR"].min(),
    "-",
    dsst_g["RIDAGEYR"].max()
)

print("\nDSST range:")
print(
    dsst_g["CFDDS"].min(),
    "-",
    dsst_g["CFDDS"].max()
)

print("\nSex:")
print(
    dsst_g["RIAGENDR"]
    .value_counts()
)

print("\nEducation:")
print(
    dsst_g["DMDEDUC2"]
    .value_counts()
    .sort_index()
)

print("\nValid wearable days:")
print(
    dsst_g["valid_days"]
    .value_counts()
    .sort_index()
)

Participants: 1289

Age range:
60.0 - 80.0

DSST range:
1.0 - 100.0

Sex:
RIAGENDR
2.0    658
1.0    631
Name: count, dtype: int64

Education:
DMDEDUC2
1.0    164
2.0    185
3.0    298
4.0    362
5.0    280
Name: count, dtype: int64

Valid wearable days:
valid_days
4      8
5      9
6     12
7     16
8    632
9    612
Name: count, dtype: int64


## 8. Preliminary Correlation Analysis

In [22]:
correlation_vars_g = [
    "CFDDS",
    "RIDAGEYR",
    "mean_daily_activity",
    "activity_sd",
    "activity_cv"
]

correlations_g = dsst_g[correlation_vars_g].corr()

correlations_g.round(3)

,CFDDS,RIDAGEYR,mean_daily_activity,activity_sd,activity_cv
CFDDS,1.000,-0.299,0.174,0.143,-0.013
RIDAGEYR,-0.299,1.000,-0.355,-0.347,-0.072
mean_daily_activity,0.174,-0.355,1.000,0.716,-0.313
activity_sd,0.143,-0.347,0.716,1.000,0.215
activity_cv,-0.013,-0.072,-0.313,0.215,1.000


In [23]:
variables_g = [
    "RIDAGEYR",
    "mean_daily_activity",
    "activity_sd",
    "activity_cv"
]

for variable in variables_g:
    temp = dsst_g[[variable, "CFDDS"]].dropna()

    r, p = pearsonr(
        temp[variable],
        temp["CFDDS"]
    )

    print(
        f"{variable}: "
        f"r = {r:.3f}, "
        f"p = {p:.4g}"
    )

RIDAGEYR: r = -0.299, p = 4.93e-28
mean_daily_activity: r = 0.174, p = 3.338e-10
activity_sd: r = 0.143, p = 2.463e-07
activity_cv: r = -0.013, p = 0.6372


In [24]:
temp = dsst_g[
    ["RIDAGEYR", "mean_daily_activity"]
].dropna()

r, p = pearsonr(
    temp["RIDAGEYR"],
    temp["mean_daily_activity"]
)

print(
    f"Age vs. mean daily activity: "
    f"r = {r:.3f}, "
    f"p = {p:.4g}"
)

Age vs. mean daily activity: r = -0.355, p = 1.275e-39


## 9. Replication Regression Models

In [25]:
dsst_g["activity_per_1000"] = (
    dsst_g["mean_daily_activity"] / 1000
)

In [26]:
model_g_1 = smf.ols(
    "CFDDS ~ RIDAGEYR + C(RIAGENDR) + C(DMDEDUC2)",
    data=dsst_g
).fit()

print(model_g_1.summary())

                            OLS Regression Results                            
Dep. Variable:                  CFDDS   R-squared:                       0.398
Model:                            OLS   Adj. R-squared:                  0.395
Method:                 Least Squares   F-statistic:                     141.3
Date:                Thu, 27 Aug 2026   Prob (F-statistic):          1.73e-137
Time:                        01:25:20   Log-Likelihood:                -5187.8
No. Observations:                1289   AIC:                         1.039e+04
Df Residuals:                    1282   BIC:                         1.043e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             69.0811      4

In [27]:
model_g_2 = smf.ols(
    """CFDDS ~ RIDAGEYR
             + C(RIAGENDR)
             + C(DMDEDUC2)
             + activity_per_1000""",
    data=dsst_g
).fit()

print(model_g_2.summary())

                            OLS Regression Results                            
Dep. Variable:                  CFDDS   R-squared:                       0.402
Model:                            OLS   Adj. R-squared:                  0.398
Method:                 Least Squares   F-statistic:                     122.8
Date:                Thu, 27 Aug 2026   Prob (F-statistic):          4.57e-138
Time:                        01:25:40   Log-Likelihood:                -5183.9
No. Observations:                1289   AIC:                         1.038e+04
Df Residuals:                    1281   BIC:                         1.043e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             61.3591      4

In [28]:
model_g_3 = smf.ols(
    """CFDDS ~ RIDAGEYR
             + C(RIAGENDR)
             + C(DMDEDUC2)
             + activity_per_1000
             + activity_cv""",
    data=dsst_g
).fit()

print(model_g_3.summary())

                            OLS Regression Results                            
Dep. Variable:                  CFDDS   R-squared:                       0.402
Model:                            OLS   Adj. R-squared:                  0.398
Method:                 Least Squares   F-statistic:                     107.5
Date:                Thu, 27 Aug 2026   Prob (F-statistic):          4.33e-137
Time:                        01:25:55   Log-Likelihood:                -5183.7
No. Observations:                1289   AIC:                         1.039e+04
Df Residuals:                    1280   BIC:                         1.043e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             59.8826      5

In [29]:
comparison_g = pd.DataFrame({
    "Model": [
        "Demographics",
        "Demographics + Activity",
        "Demographics + Activity + Variability"
    ],
    "R_squared": [
        model_g_1.rsquared,
        model_g_2.rsquared,
        model_g_3.rsquared
    ],
    "Adjusted_R_squared": [
        model_g_1.rsquared_adj,
        model_g_2.rsquared_adj,
        model_g_3.rsquared_adj
    ],
    "AIC": [
        model_g_1.aic,
        model_g_2.aic,
        model_g_3.aic
    ]
})

comparison_g.round(4)

,Model,R_squared,Adjusted_R_squared,AIC
0,Demographics,0.3980,0.3952,10389.6783
1,Demographics + Activity,0.4016,0.3984,10383.8861
2,Demographics + Activity + Variability,0.4018,0.3981,10385.4939
